# HyperView: CLIP vs HyCoCLIP with 2D UMAP Projection on CIFAR-100

This notebook compares two embedding spaces on the same image set.

- [**CLIP**](https://github.com/openai/CLIP) maps images and text into one shared vector space.
    
- [**HyCoCLIP**](https://github.com/PalAvik/hycoclip) maps images into a space designed for hierarchy and tree like structure.
    
- We then compute a **2D projection** of each embedding space with [UMAP](https://umap-learn.readthedocs.io/en/latest/) so we can inspect clusters and label structure.
    

HyperView allows us to compare the embedding spaces of these different geometries.

In this demo, we use the [CIFAR-100](https://huggingface.co/datasets/uoft-cs/cifar100) dataset of tiny images, available through HuggingFace.

## Install

HyperView is a library for dataset curation and model analysis. It provides tools for interactive visualization of the embedding space.

In [4]:
%%capture
# If you run this in Google Colab, install HyperView with this command.
!uv pip install hyperview

## Import HyperView


We import HyperView and use it as the main interface for:

- reading data
    
- computing embeddings with pretrained models
    
- computing a 2D projection for visualization
    
- launching the interactive viewer
    

In [5]:
import hyperview as hv

## Configuration

We specify the dataset and model settings in a single dictionary.

### Data source

- We load **CIFAR-100** from Hugging Face.
    
- We use the **test** split.
    
- `img` is the image field.
    
- We use `coarse_label` for labels.

Why coarse labels:

- HyperView disables distinct label coloring when there are more than 20 labels.
    
- CIFAR-100 has 100 fine labels, but 20 coarse labels.
    
- Using coarse labels keeps label coloring useful in the viewer.
    

### Sampling

- `NUM_SAMPLES = 200` keeps embedding and layout computation fast enough for a demo.
    
- Increase this if you want denser clusters, but expect more compute time.
    

### Models

- `openai/clip-vit-base-patch32` is a common CLIP baseline.
    
- `hycoclip-vit-s` is a HyCoCLIP model that targets hierarchical structure.

In [6]:
DATASET_NAME = "cifar100_coarse_clip_hyper_models"
HF_DATASET = "uoft-cs/cifar100"
HF_SPLIT = "test"
HF_IMAGE_KEY = "img"
# NOTE: HyperView disables distinct label coloring when there are >20 labels.
# CIFAR-100 has 100 fine labels, but only 20 coarse labels.
HF_LABEL_KEY = "coarse_label"
NUM_SAMPLES = 200
CLIP_MODEL_ID = "openai/clip-vit-base-patch32"
HYPER_MODELS_MODEL_ID = "hycoclip-vit-s"

## Load CIFAR-100 into a HyperView dataset

We create a HyperView `Dataset` object and then import samples from Hugging Face.

Key parameters:

- `persist=False` keeps this dataset in memory for this run.
    
- `max_samples=NUM_SAMPLES` subsamples the split.
    

At the end, `len(dataset)` is the number of loaded examples.

In [7]:
print("Loading CIFAR-100 from Hugging Face...")
dataset = hv.Dataset(DATASET_NAME, persist=False)
dataset.add_from_huggingface(
    HF_DATASET,
    split=HF_SPLIT,
    image_key=HF_IMAGE_KEY,
    label_key=HF_LABEL_KEY,
    max_samples=NUM_SAMPLES,
)
print(f"Loaded {len(dataset)} samples")



Loading CIFAR-100 from Hugging Face...
Loading 200 samples from uoft-cs/cifar100...
Images saved to: /root/.hyperview/media/huggingface/uoft-cs_cifar100/test
Loaded 200 samples


## Compute CLIP embeddings

### What CLIP embeddings represent

CLIP trains an image encoder and a text encoder so that matching image text pairs have nearby vectors. When we compute image embeddings here, each image becomes a single vector in a shared space that also supports text vectors.

### What we do in code

- `compute_embeddings(CLIP_MODEL_ID)` runs the CLIP image encoder and stores one vector per image.
    
- The returned `space_key` is a handle to that embedding space inside HyperView.
    

### Why we compute a 2D visualization

High dimensional vectors are hard to inspect. We compute a 2D layout so we can see neighborhood structure.

- `compute_visualization(..., geometry="euclidean")` treats the embedding space as Euclidean.
    
- This matches how CLIP vectors are often used with cosine similarity or dot product in a flat vector space.
    

Note on “UMAP”  
This notebook focuses on a 2D projection step. HyperView computes a 2D layout for the viewer. The title calls this UMAP. In practice, you should treat this as a nonlinear projection that preserves local neighborhoods better than PCA.

## Compute HyCoCLIP embeddings

### Why a different geometry

Some datasets have hierarchical label structure. A tree structure is hard to represent in Euclidean space without distortion.

Hyperbolic spaces can represent tree growth patterns with less distortion than Euclidean spaces. A common model is the **Poincaré ball**, which is a way to work with hyperbolic geometry in a bounded region.

### What we do in code

- `compute_embeddings(model=HYPER_MODELS_MODEL_ID)` computes embeddings from the HyCoCLIP model.
    
- `compute_visualization(..., geometry="poincare")` tells HyperView to treat distances and neighborhoods using Poincaré geometry.
    

How to read the plot

- In Poincaré style layouts, points near the center and points near the boundary can have different distance behavior than Euclidean layouts.
    
- Focus on neighborhood membership and cluster separation rather than raw coordinate scale.

In [14]:
clip_space = dataset.compute_embeddings(CLIP_MODEL_ID)
clip_space

All 200 samples already have embeddings in space 'embed-anything__openai_clip-vit-base-patch32__4771034973d8'


'embed-anything__openai_clip-vit-base-patch32__4771034973d8'

In [10]:
hyperbolic_clip_space = dataset.compute_embeddings(model=HYPER_MODELS_MODEL_ID)
hyperbolic_clip_space

Computing embeddings for 200 samples...


'hyper-models__hycoclip-vit-s__b63e9ee38a30'

## UMAP Parameters: `n_neighbors` and `min_dist`

UMAP creates 2D visualizations through a two-phase process:

1. **Local structure learning** (controlled by `n_neighbors`)
2. **Layout optimization** (controlled by `min_dist`)

These parameters have different effects depending on whether you're working with Euclidean or hyperbolic embeddings.


## `n_neighbors`: Balancing Local vs Global Structure

The `n_neighbors` parameter determines how many nearby points UMAP considers when learning the manifold structure.

**Small values (5-15):**
- UMAP focuses on immediate neighbors
- Preserves fine-grained local structure
- Creates tight, distinct clusters
- May fragment large-scale patterns
- Better for finding small subgroups

**Large values (30-100):**
- UMAP considers broader neighborhoods
- Captures global dataset organization
- Creates more connected layouts
- May blur boundaries between clusters
- Better for understanding overall structure

**Technical detail:** UMAP constructs a k-nearest neighbor graph using this parameter. The graph encodes which points should remain close in the 2D projection.

**Default: 15**, which is a middle ground that works for most datasets

## `min_dist`: Controlling Point Spacing

The `min_dist` parameter sets the minimum allowed distance between points in the final 2D layout. This is the distance that UMAP forces between **all points** in the 2D layout. When `min-dist` is large points that are in the samle cluster are pushed apart from each other. The visual effect is that we see more empty space everywhere in the plot.

**Small values (0.0-0.1):**
- Points can be placed very close together
- Clusters appear dense and compact
- Easier to see cluster boundaries
- Can create overlapping point clouds, making them difficult to select with HyperView's lasso tool
- Good for large datasets where you want to see density

**Large values (0.3-0.99):**
- Forces points to spread out
- Individual points are more visible and the layout appears less crowded
- May exaggerate distances between similar items
- Good for small datasets or when you need to see each point

**Technical detail:** This parameter affects the "attractive force" in UMAP's  layout.

**Default: 0.1** (high density some clumping of embedding projections)

## Parameter Interaction in UMAP

The two parameters work together to shape your visualization:

**n_neighbors + min_dist together:**

| n_neighbors | min_dist | Result |
|------------|----------|--------|
| Small (5)  | Small (0.01) | Many tight, separated micro-clusters |
| Small (5)  | Large (0.5)  | Fragmented layout with forced spacing |
| Large (50) | Small (0.01) | Dense, continuous manifold structure |
| Large (50) | Large (0.5)  | Smooth, evenly distributed layout |

**Practical example:**
```python
# For finding fine subgroups in cell types
compute_visualization(n_neighbors=10, min_dist=0.05)

# For understanding broad relationships in a corpus
compute_visualization(n_neighbors=50, min_dist=0.3)

## Parameter Interaction in UMAP

The two parameters work together to shape your visualization:

**n_neighbors + min_dist together:**

| n_neighbors | min_dist | Result |
|------------|----------|--------|
| Small (5)  | Small (0.01) | Many tight, separated and small clusters |
| Small (5)  | Large (0.5)  | Fragmented layout with forced spacing |
| Large (50) | Small (0.01) | Dense, continuous structure |
| Large (50) | Large (0.5)  | Smooth, evenly distributed layout |

**Snippets:**
```python
# For finding fine subgroups in cell types
compute_visualization(n_neighbors=10, min_dist=0.05)

# For understanding broad relationships in a corpus
compute_visualization(n_neighbors=50, min_dist=0.3)





### Euclidean UMAP (Standard Embeddings)

```markdown
## UMAP on Euclidean Embeddings

For standard models (CLIP, ResNet, etc.), embeddings live in flat Euclidean space.

**What happens:**
1. UMAP measures distances using the specified metric (default: cosine)
2. Constructs a neighbor graph in the high-dimensional space
3. Optimizes a 2D layout in flat Euclidean space
4. Output coordinates are (x, y) pairs with no geometric constraints (can be as far away from each other as originally computed)

**Metric choice matters:**
- `metric="cosine"`: Best for normalized embeddings (CLIP, sentence transformers)
- `metric="euclidean"`: Best for embeddings where magnitude matters
- `metric="manhattan"`: Sometimes better for sparse or categorical data

In [9]:
dataset.compute_visualization(space_key=clip_space, # Which embedding space to project
                              method="umap", # Projection method (only 'umap' supported)
                              geometry="euclidean", # Output geometry: 'euclidean' or 'poincare'
                              n_neighbors=15, #  UMAP: Number of neighbors (default: 15)
                              min_dist=0.1,  #  UMAP: Minimum distance (default: 0.1)
                              metric="cosine", # UMAP: Distance metric (default: 'cosine')
                              force=True) # Force recomputation if layout exists


Computing euclidean umap layout for 200 samples...


'embed-anything__openai_clip-vit-base-patch32__4771034973d8__euclidean_umap_92b543de'

### Cell 6: Hyperbolic UMAP (HyCoCLIP and Hyperboloid Embeddings)

```markdown
## UMAP on Hyperbolic Embeddings

For hyperbolic models (HyCoCLIP), embeddings live in curved hyperbolic space (hyperboloid model).

**What happens:**
1. HyperView converts hyperboloid coordinates to the Poincaré ball (unit disk)
2. UMAP measures distances using Poincaré distance (hyperbolic geometry)
3. Optimizes a 2D layout using hyperbolic distance in the output space
4. Output coordinates are (x, y) pairs **constrained to the unit disk**

**Key differences from Euclidean:**
- Distance metric is **automatically** "poincare" (you cannot override this)
- Points near the disk center have more "room" (hyperbolic space expands near edges)
- Same Euclidean distance means different hyperbolic distances at different radii
- Natural hierarchy: broader or more ambiguous concepts toward center, specific items toward edges. Mislabeled examples also appear near the center as they are hard to separate.

**Parameter effects:**
- `n_neighbors`: Works the same as Euclidean (controls local vs global)
- `min_dist`: Interprets distances **in hyperbolic geometry**
  - Same numeric value creates different visual spacing than in Euclidean
  - Points near the boundary can appear closer in visual distance but farther in hyperbolic distance

**Example:**
```python
# Hyperbolic CLIP embeddings
dataset.compute_embeddings(model="hycoclip-vit-s")  # Creates hyperboloid space
dataset.compute_visualization(
    geometry="poincare",   # Output to Poincaré disk
    # metric is automatically "poincare" (ignores what you pass)
    n_neighbors=15,
    min_dist=0.1
)

In [15]:
dataset.compute_visualization(space_key=hyperbolic_clip_space,
                              method="umap",
                              geometry="poincare", # Use Poincare ball method to project hyperbolic embedding to 2D plane
                              # metric is overriden to "poincare", see https://github.com/Hyper3Labs/HyperView/blob/main/src/hyperview/embeddings/projection.py#L102
                              n_neighbors=15,
                              min_dist=0.1,
                              )

Computing poincare umap layout for 200 samples...


'hyper-models__hycoclip-vit-s__b63e9ee38a30__poincare_umap_92b543de'

## Launch the interactive app in HyperView

`hv.launch(dataset, open_browser=True)` starts a local server and opens the viewer.

In the UI, you should be able to:

- switch between embedding spaces (CLIP vs HyCoCLIP)
    
- inspect image thumbnails for selected points
    
- compare how coarse labels cluster under each geometry
    


In [ ]:
hv.launch(dataset, open_browser=True)

## Euclidean vs Poincaré: Visual Differences

**Euclidean geometry (`geometry="euclidean"`):**
- Points spread across an unbounded 2D plane
- Distance is uniform everywhere (1 unit = 1 unit, regardless of location)
- Typical output: points in a square/rectangular region
- Best for: Datasets without strong hierarchical structure

**Poincaré geometry (`geometry="poincare"`):**
- Points constrained to a unit disk (circle with radius 1)
- Distance is **non-uniform**: more space near edges, compressed near center
- Typical output: hierarchical organization
  - General/abstract concepts cluster near the center
  - Specific/detailed items pushed toward the boundary
- Best for: Datasets with natural hierarchies (taxonomies, concept trees)

**When to use Poincaré output:**
1. You have hyperbolic embeddings (HyCoCLIP)
2. Your data has hierarchical structure (animal taxonomy, document topics)
3. You want to visualize relationships at multiple scales simultaneously

**When to use Euclidean output:**
1. You have standard embeddings (CLIP, ResNet)
2. Your data has flat or network structure
3. You want familiar, intuitive spatial relationships



## Suggested checks

- Are nearest neighbors under Euclidean CLIP similar to nearest neighbors under Poincaré HyCoCLIP?


    